# cfDNA Fragmentomics — Exploratory Data Analysis

**Goal:** Build confidence that a real biological signal exists in the data *before* training any model.  
If EDA looks wrong — fix the data, not the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys, pathlib

ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT))

df = pd.read_csv(ROOT / 'data/processed/features_model.csv')
print('Shape:', df.shape)
print(df['label'].value_counts())

---
## Step 1 — Class Distribution

**Why this matters:**  
You have ~1.78× more healthy samples than cancer. This is a class imbalance.  

A naive model that *always predicts healthy* would score **64% accuracy** — and be completely useless for detecting cancer. Accuracy is the wrong metric here. This is why we use **AUC-ROC** and why models need `class_weight='balanced'` or equivalent.

Check this first. Always.

In [ ]:
counts = df['label'].value_counts()
ratio  = counts['healthy'] / counts['cancer']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values,
              color=['#2196F3', '#F44336'], width=0.5, edgecolor='white')

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of samples')
ax.set_ylim(0, counts.max() * 1.15)
ax.spines[['top','right']].set_visible(False)
ax.text(0.98, 0.95, f'Imbalance ratio: {ratio:.2f}:1',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='gray')
plt.tight_layout()
plt.savefig(ROOT / 'results/eda_01_class_distribution.png', dpi=150)
plt.show()

print(f'Healthy : {counts["healthy"]}  ({100*counts["healthy"]/len(df):.1f}%)')
print(f'Cancer  : {counts["cancer"]}  ({100*counts["cancer"]/len(df):.1f}%)')
print(f'Ratio   : {ratio:.2f}:1')
print()
print('Implication: use AUC-ROC as primary metric, not accuracy.')
print('All models will need class_weight="balanced" or equivalent.')